In [1]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

merged_df= merged_df.drop(columns=["flag_document_5","organization_type_Industry: type 5","bureau_balance_potential_on_going_loan_loan_1","bureau_amt_credit_sum_limit_long_limit_loan_1","bureau_balance_status_score_max_loan_1","flag_document_11","own_car_age_is_missing","bureau_amt_credit_sum_limit_long_limit_loan_2"]) #,"building_score_std" #


#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

#feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "reference_for_purr.csv")
#X= clean_noise_from_feature_importance(feature_raper,X,0.0000001)
inhert_features = ['bureau_amt_credit_sum_overdue_is_missing_loan_1', 'flag_document_14', 'organization_type_Transport: type 2', 'organization_type_Industry: type 1', 'organization_type_Other trade', 'organization_type_Advertising', 'bureau_days_credit_enddate_is_missing_loan_2', 'bureau_credit_active_sold_loan_2', 'bureau_days_credit_enddate_is_missing_loan_1', 'closed_credit_active_sold_closed_sum', 'organization_type_Industry: type 2', 'bureau_amt_credit_sum_overdue_is_missing_loan_2', 'amt_req_credit_berau_hour', 'organization_type_Industry: type 7', 'flag_cont_mobile', 'organization_type_Realtor', 'organization_type_Telecom', 'ext_source_3_is_missing', 'organization_type_Services', 'organization_type_Electricity', 'organization_type_Restaurant', 'bureau_cnt_credit_prolong_loan_2', 'organization_type_Housing', 'sellerplace_area_is_missing_prev_1', 'amt_goods_price_is_missing', 'flag_document_9', 'ext_source_1_is_missing', 'organization_type_Industry: type 4', 'organization_type_Other', 'flag_document_13', 'organization_type_Agriculture', 'bureau_credit_active_sold_loan_1', 'organization_type_Other industry', 'flag_last_application_in_day_prev_1', 'closed_amt_credit_sum_limit_closed_min', 'organization_type_Industry: type 12', 'organization_type_Cleaning', 'bureau_days_credit_enddate_fourth_positive_cluster_loan_1', 'instalments_dead_tail_length_prev_1', 'organization_type_Trade: type 6', 'organization_type_Postal', 'info_of_social_circule_is_missing', 'bureau_amt_credit_sum_debt_is_negative_loan_1', 'flag_document_16', 'bureau_cnt_credit_prolong_loan_1', 'organization_type_Industry: type 11', 'organization_type_Emergency', 'amt_req_credit_breau_week', 'organization_type_Insurance', 'bureau_amt_credit_sum_debt_is_negative_loan_2', 'closed_credit_active_sold_closed_mean', 'ext_source_2_is_missing', 'organization_type_Transport: type 4', 'organization_type_Trade: type 2', 'flag_own_car', 'organization_type_Security']
high_correlation_features = ['flag_emp_phone', 'organization_type_XNA', 'bureau_days_credit_enddate_closed_loan_2', 'bureau_balance_is_delincuency_sum_loan_1', 'bureau_balance_status_score_max_loan_2', 'bureau_days_enddate_fact_is_missing_loan_1', 'bureau_credit_active_active_loan_1', 'bureau_credit_active_closed_loan_2', 'bureau_credit_active_active_loan_2', 'active_have_amt_credit_sum_overdue_active_sum', 'closed_amt_credit_sum_debt_closed_sum', 'amt_down_payment_is_missing_prev_1', 'rate_down_payment_is_missing_prev_1', 'rate_interesting_is_missing_prev_1', 'instalments_extra_instalament_sum_prev_1', 'rate_down_payment_is_missing_mean', 'rate_down_payment_is_missing_sum', 'amt_down_payment_is_missing_sum', 'instalments_dead_tail_length_max', 'bureau_balance_months_since_delincuency_loan_2', 'active_balance_months_since_delincuency_active_max', 'bureau_has_bureau_balance_data_loan_1', 'bureau_amt_credit_sum_limit_is_missing_loan_2', 'bureau_days_credit_enddate_first_positive_cluster_loan_2', 'bureau_credit_currency_loan_2', 'bureau_days_credit_enddate_second_positive_cluster_loan_2', 'instalments_repeated_for_underpayment_sum_prev_1', 'bureau_credit_currency_loan_1', 'bureau_days_credit_enddate_second_positive_cluster_loan_1', 'client_without_querys', 'closed_balance_status_score_max_closed_max', 'days_last_due_has_sentinel_value_prev_1', 'name_goods_category_prev_1', 'emergencystate_mode', 'days_and_insurance_information_are_missing_prev_1', 'days_first_drawing_has_sentinel_value_prev_1', 'instalments_potentially_on_going_prev_1', 'flag_invalid_surface_sellerplace_area_prev_1', 'amt_goods_price_is_missing_prev_1', 'flag_region_not_work']
#X= X.drop(columns=["flag_phone", "family_status","bureau_amt_credit_sum_limit_is_missing_loan_2", *inhert_features])
X= X.drop(columns=['closed_days_credit_closed_max', *inhert_features, *high_correlation_features])



baseline_oof_auc, baseline_std = run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"long-road-basura-pura+correlation")


🏃 View run long-road-basura-pura_child_1 at: http://localhost:5332/#/experiments/3/runs/10f915cfda9d4eb5a7ae442d3c7bac41
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-basura-pura_child_2 at: http://localhost:5332/#/experiments/3/runs/7d36fdb89d014b928b6f46ea1c8b4310
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-basura-pura_child_3 at: http://localhost:5332/#/experiments/3/runs/5b49d4b9988f47cbb907d96a83ee9851
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-basura-pura_child_4 at: http://localhost:5332/#/experiments/3/runs/4c7932f2504345b598458cdb74ecda76
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-basura-pura_child_5 at: http://localhost:5332/#/experiments/3/runs/1070f69f7c284311a0869973ec7a7a15
🧪 View experiment at: http://localhost:5332/#/experiments/3
AUC per fold= 0.781 ± 0.001(std), auc_score_OOF= 0.781 result of CV with 5 folds. 
🏃 View run P